# ============================================
#  Notebook 03 — EDA, Classification & Diagnostic Metrics
#  Memorial Sloan Kettering | Goel Lab
# ============================================

# Notebook 3: EDA, Classification & Diagnostic Metrics, Inference

**Sections:**
1. EDA — Average total correct vs omitted vs fabricated at observation level, grouped by feature
2. Classification & diagnostic metrics with faceted plots (Radiology vs Pathology)
3. All tables and plots from `main_analysis.py` reproduced in notebook form
4. Inference — one-sided McNemar p-values at observation level, grouped by feature, stratified by domain

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import binomtest
from sklearn.metrics import roc_curve, auc, precision_recall_curve

pd.set_option('display.float_format', lambda x: '%.4f' % x)
sns.set_style('whitegrid')

PROJECT_ROOT = Path("/Users/robertjames/Documents/llm_summarization")
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "merged_llm_summary_validation_datasheet_deidentified.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from metrics_utils import (
    compute_confusion_counts, compute_metrics_from_counts,
    bootstrap_ci, element_metric_pvalue,
    binary_predictions_from_annotator, roc_pr_from_binary,
    build_confusion_df_from_counts, mcnemar_exact_from_masks,
    metric_correct_masks,
)

In [ ]:
data = pd.read_excel(DATA_PATH)
string_data = ["NA", "na", "n/a", "N/A", "NA ", " na", " n/a", " N/A"]
data = data.replace(string_data, "N/A")
print(f"Dataset: {data.shape[0]} observations x {data.shape[1]} columns")

In [ ]:
ELEMENTS = {
    "Lesion Size": {"source": "lesion_size_status_source", "human": "lesion_size_status_human", "ai": "lesion_size_status_ai"},
    "Lesion Laterality": {"source": "laterality_status_source", "human": "laterality_status_human", "ai": "laterality_status_ai"},
    "Lesion Location": {"source": "lesion_location_status_source", "human": "lesion_location_status_human", "ai": "lesion_location_status_ai"},
    "Calcifications / Asymmetry": {"source": "calcifications_asymmetry_status_source", "human": "calcifications_asymmetry_status_human", "ai": "calcifications_asymmetry_status_ai"},
    "Additional Enhancement (MRI)": {"source": "additional_enhancement_mri_status_source", "human": "additional_enhancement_mri_status_human", "ai": "additional_enhancement_mri_status_ai"},
    "Extent": {"source": "extent_status_source", "human": "extent_status_human", "ai": "extent_status_ai"},
    "Accurate Clip Placement": {"source": "accurate_clip_placement_status_source", "human": "accurate_clip_placement_status_human", "ai": "accurate_clip_placement_status_ai"},
    "Workup Recommendation": {"source": "workup_recommendation_status_source", "human": "workup_recommendation_status_human", "ai": "workup_recommendation_status_ai"},
    "Lymph Node": {"source": "Lymph node_status_source", "human": "Lymph node_status_human", "ai": "Lymph node_status_ai"},
    "Chronology Preserved": {"source": "chronology_preserved_status_source", "human": "chronology_preserved_status_human", "ai": "chronology_preserved_status_ai"},
    "Biopsy Method": {"source": "biopsy_method_status_source", "human": "biopsy_method_status_human", "ai": "biopsy_method_status_ai"},
    "Invasive Component Size (Pathology)": {"source": "invasive_component_size_pathology_status_source", "human": "invasive_component_size_pathology_status_human", "ai": "invasive_component_size_pathology_status_ai"},
    "Histologic Diagnosis": {"source": "histologic_diagnosis_status_source", "human": "histologic_diagnosis_status_human", "ai": "histologic_diagnosis_status_ai"},
    "Receptor Status": {"source": "receptor_status_source", "human": "receptor_status_human", "ai": "receptor_status_ai"},
}
DOMAINS = {
    "Radiology": ["Lesion Size", "Lesion Laterality", "Lesion Location",
                  "Calcifications / Asymmetry", "Additional Enhancement (MRI)",
                  "Extent", "Accurate Clip Placement", "Workup Recommendation",
                  "Lymph Node", "Chronology Preserved", "Biopsy Method"],
    "Pathology": ["Biopsy Method", "Invasive Component Size (Pathology)",
                  "Histologic Diagnosis", "Receptor Status"],
}
BOOTSTRAP_METRICS = ["accuracy", "sensitivity", "specificity", "ppv", "npv",
                     "precision", "recall", "fabrication_rate", "f1"]

---
## Part 1: EDA — Correct vs Omitted vs Fabricated at Observation Level

### 1.1 Count correct (1), omitted (2), fabricated (3) per observation per feature

In [ ]:
obs_rows = []
for element, cols in ELEMENTS.items():
    s_col, h_col, a_col = cols["source"], cols["human"], cols["ai"]
    for idx, row in data.iterrows():
        source_val = row.get(s_col)
        for annotator, ann_col in [("Human", h_col), ("AI", a_col)]:
            ann_val = row.get(ann_col)
            if pd.isna(source_val) or pd.isna(ann_val):
                status = "missing"
            elif source_val == 1 and ann_val == 1:
                status = "correct"
            elif source_val == 1 and ann_val == 2:
                status = "omitted"
            elif source_val == 1 and ann_val == 3:
                status = "fabricated"
            elif source_val == 0 and ann_val == "N/A":
                status = "correct"  # TN
            else:
                status = "other"
            obs_rows.append({"obs_idx": idx, "Element": element, "Annotator": annotator, "status": status})

df_obs = pd.DataFrame(obs_rows)
print(f"Observation-level status records: {len(df_obs)}")

In [ ]:
# Summary: total correct, omitted, fabricated per element per annotator
eda_summary = (
    df_obs[df_obs["status"].isin(["correct", "omitted", "fabricated"])]
    .groupby(["Element", "Annotator", "status"])
    .size()
    .reset_index(name="count")
)

# Pivot for display
eda_pivot = eda_summary.pivot_table(index=["Element", "Annotator"], columns="status",
                                     values="count", fill_value=0).reset_index()
eda_pivot = eda_pivot[["Element", "Annotator", "correct", "omitted", "fabricated"]]
print("Correct / Omitted / Fabricated per Element per Annotator")
eda_pivot

### 1.2 Faceted Bar Charts — Correct vs Omitted vs Fabricated by Feature (Rad vs Path)

In [ ]:
for domain_name, domain_elements in DOMAINS.items():
    df_dom = eda_pivot[eda_pivot["Element"].isin(domain_elements)].copy()
    n_elements = len(domain_elements)
    fig, axes = plt.subplots(1, 3, figsize=(6 * 3, max(5, n_elements * 0.5)))
    fig.suptitle(f"{domain_name} Features: Correct vs Omitted vs Fabricated",
                 fontsize=16, fontweight="bold", y=1.02)

    for ax_i, metric in enumerate(["correct", "omitted", "fabricated"]):
        ax = axes[ax_i]
        sub = df_dom[["Element", "Annotator", metric]].copy()
        sub_pivot = sub.pivot(index="Element", columns="Annotator", values=metric).reindex(domain_elements)
        x = np.arange(len(sub_pivot))
        width = 0.35
        ax.barh(x - width/2, sub_pivot["Human"], width, label="Human", color="#2c3e50")
        ax.barh(x + width/2, sub_pivot["AI"], width, label="AI", color="#95a5a6")
        ax.set_yticks(x)
        ax.set_yticklabels(sub_pivot.index, fontsize=9)
        ax.set_xlabel("Count")
        ax.set_title(metric.title(), fontsize=14, fontweight="bold")
        ax.legend(loc="lower right")
        ax.grid(axis="x", linestyle=":", alpha=0.6)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"eda_{domain_name.lower()}_correct_omitted_fabricated.png", dpi=300, bbox_inches="tight")
    plt.show()

### 1.3 Stacked Histogram — Distribution of Correct/Omitted/Fabricated per Observation

In [ ]:
# Per-observation totals
obs_totals = (
    df_obs[df_obs["status"].isin(["correct", "omitted", "fabricated"])]
    .groupby(["obs_idx", "Annotator", "status"])
    .size()
    .reset_index(name="count")
    .pivot_table(index=["obs_idx", "Annotator"], columns="status", values="count", fill_value=0)
    .reset_index()
)

for annotator in ["Human", "AI"]:
    sub = obs_totals[obs_totals["Annotator"] == annotator]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist([sub["correct"], sub["omitted"], sub["fabricated"]],
            bins=range(0, 16), stacked=True, label=["Correct", "Omitted", "Fabricated"],
            color=["#27ae60", "#f39c12", "#e74c3c"], edgecolor="black", alpha=0.85)
    ax.set_xlabel("Count per Observation")
    ax.set_ylabel("Number of Observations")
    ax.set_title(f"{annotator}: Distribution of Correct/Omitted/Fabricated per Observation",
                 fontsize=13, fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"eda_obs_histogram_{annotator.lower()}.png", dpi=300)
    plt.show()

---
## Part 2: Classification & Diagnostic Metrics (from main_analysis.py)

### 2.1 Element-Level Metrics

In [ ]:
element_rows = []
for element_name, cols in ELEMENTS.items():
    source_col, human_col, ai_col = cols["source"], cols["human"], cols["ai"]
    mask = data[[source_col, human_col, ai_col]].notnull().all(axis=1)
    d = data.loc[mask].copy()
    if d.empty:
        continue

    human_counts = compute_confusion_counts(d, source_col, human_col)
    ai_counts = compute_confusion_counts(d, source_col, ai_col)
    human_metrics = compute_metrics_from_counts(**human_counts)
    ai_metrics = compute_metrics_from_counts(**ai_counts)

    # Bootstrap CIs
    ci_h, ci_ai = {}, {}
    for m in BOOTSTRAP_METRICS:
        ci_h[m] = bootstrap_ci(d, source_col, human_col, m, n_boot=2000)
        ci_ai[m] = bootstrap_ci(d, source_col, ai_col, m, n_boot=2000)

    # McNemar p-values
    p_vals = {}
    for m in ["accuracy", "sensitivity", "specificity", "ppv", "npv", "fabrication_rate"]:
        p_vals[m] = element_metric_pvalue(d, source_col, human_col, ai_col, metric_name=m)

    for annotator, counts, metrics, ci in [("Human", human_counts, human_metrics, ci_h),
                                            ("AI", ai_counts, ai_metrics, ci_ai)]:
        row = {"Element": element_name, "Annotator": annotator}
        row.update({k: v for k, v in counts.items()})
        for m in BOOTSTRAP_METRICS:
            row[m] = metrics[m]
            row[f"{m}_ci_low"] = ci[m][0]
            row[f"{m}_ci_high"] = ci[m][1]
        for m, pv in p_vals.items():
            row[f"p_mcnemar_{m}"] = pv
        element_rows.append(row)

df_elements = pd.DataFrame(element_rows)
df_elements.to_csv(OUTPUT_DIR / "element_level_metrics.csv", index=False)
print(f"Element-level metrics: {df_elements.shape}")
df_elements.head(4)

### 2.2 Confusion Matrix Heatmaps

In [ ]:
from metrics_utils import plot_confusion_heatmap

# Aggregate confusion across all elements
agg_counts = {"Human": {"TP": 0, "FP": 0, "FN": 0, "TN": 0},
              "AI": {"TP": 0, "FP": 0, "FN": 0, "TN": 0}}
for _, row in df_elements.iterrows():
    ann = row["Annotator"]
    for k in ["TP", "FP", "FN", "TN"]:
        agg_counts[ann][k] += int(row[k])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, ann in zip(axes, ["Human", "AI"]):
    c = agg_counts[ann]
    cm_df = build_confusion_df_from_counts(c["TP"], c["FP"], c["FN"], c["TN"])
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", ax=ax, linewidths=1)
    ax.set_title(f"{ann} — Aggregate Confusion Matrix", fontsize=13, fontweight="bold")
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_heatmaps.png", dpi=300, bbox_inches="tight")
plt.show()

### 2.3 Faceted Diagnostic Metrics: Radiology vs Pathology

In [ ]:
diag_metrics = ["accuracy", "sensitivity", "specificity", "ppv", "npv", "fabrication_rate"]

for domain_name, domain_elements in DOMAINS.items():
    df_dom = df_elements[df_elements["Element"].isin(domain_elements)].copy()
    n_metrics = len(diag_metrics)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"{domain_name}: Diagnostic Metrics — Human vs AI (with 95% CIs)",
                 fontsize=16, fontweight="bold", y=1.02)
    axes = axes.flatten()

    for ax_idx, metric in enumerate(diag_metrics):
        ax = axes[ax_idx]
        human = df_dom[df_dom["Annotator"] == "Human"].set_index("Element").reindex(domain_elements)
        ai = df_dom[df_dom["Annotator"] == "AI"].set_index("Element").reindex(domain_elements)

        x = np.arange(len(domain_elements))
        width = 0.35
        h_vals = human[metric].values.astype(float)
        a_vals = ai[metric].values.astype(float)

        h_err = np.array([
            np.clip(h_vals - human[f"{metric}_ci_low"].values.astype(float), 0, None),
            np.clip(human[f"{metric}_ci_high"].values.astype(float) - h_vals, 0, None),
        ])
        a_err = np.array([
            np.clip(a_vals - ai[f"{metric}_ci_low"].values.astype(float), 0, None),
            np.clip(ai[f"{metric}_ci_high"].values.astype(float) - a_vals, 0, None),
        ])

        ax.bar(x - width/2, h_vals, width, label="Human", color="#2c3e50")
        ax.bar(x + width/2, a_vals, width, label="AI", color="#95a5a6")
        ax.errorbar(x - width/2, h_vals, yerr=h_err, fmt="none", ecolor="black", capsize=3)
        ax.errorbar(x + width/2, a_vals, yerr=a_err, fmt="none", ecolor="black", capsize=3)

        # Add significance stars
        p_col = f"p_mcnemar_{metric}"
        if p_col in human.columns:
            for i, elem in enumerate(domain_elements):
                pval = human.loc[elem, p_col] if elem in human.index else np.nan
                if pd.notna(pval) and pval < 0.05:
                    star = "***" if pval < 0.001 else ("**" if pval < 0.01 else "*")
                    y_max = max(h_vals[i], a_vals[i]) + max(h_err[1, i], a_err[1, i]) + 0.03
                    ax.text(x[i], min(y_max, 1.08), star, ha="center", va="bottom", fontsize=11)

        ax.set_xticks(x)
        ax.set_xticklabels(domain_elements, rotation=45, ha="right", fontsize=8)
        ax.set_ylim(0, 1.15)
        ax.set_title(metric.replace("_", " ").title(), fontsize=13, fontweight="bold")
        ax.grid(axis="y", linestyle=":", alpha=0.6)
        if ax_idx == 0:
            ax.legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"faceted_diagnostic_{domain_name.lower()}.png", dpi=300, bbox_inches="tight")
    plt.show()

### 2.4 Side-by-Side Bar: Total Correct / Omitted / Fabricated per Feature (Faceted by Domain)

In [ ]:
for domain_name, domain_elements in DOMAINS.items():
    df_dom_eda = eda_pivot[eda_pivot["Element"].isin(domain_elements)].copy()
    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(domain_elements) * 0.6)),
                              sharey=True)
    fig.suptitle(f"{domain_name}: Total Counts per Feature — Human vs AI",
                 fontsize=15, fontweight="bold", y=1.02)

    for ax_i, annotator in enumerate(["Human", "AI"]):
        ax = axes[ax_i]
        sub = df_dom_eda[df_dom_eda["Annotator"] == annotator].set_index("Element").reindex(domain_elements)
        y = np.arange(len(domain_elements))
        width = 0.25
        ax.barh(y - width, sub["correct"], width, label="Correct", color="#27ae60")
        ax.barh(y, sub["omitted"], width, label="Omitted", color="#f39c12")
        ax.barh(y + width, sub["fabricated"], width, label="Fabricated", color="#e74c3c")
        ax.set_yticks(y)
        ax.set_yticklabels(domain_elements, fontsize=9)
        ax.set_xlabel("Count")
        ax.set_title(annotator, fontsize=13, fontweight="bold")
        ax.legend(loc="lower right")
        ax.grid(axis="x", linestyle=":", alpha=0.6)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"eda_counts_{domain_name.lower()}_human_vs_ai.png", dpi=300, bbox_inches="tight")
    plt.show()

---
## Part 3: Domain-Level Aggregated Metrics

In [ ]:
domain_rows = []
for domain_name, domain_elements in DOMAINS.items():
    for annotator in ["Human", "AI"]:
        sub = df_elements[(df_elements["Annotator"] == annotator) &
                          (df_elements["Element"].isin(domain_elements))]
        agg = {"Domain": domain_name, "Annotator": annotator}
        agg["TP"] = int(sub["TP"].sum())
        agg["FP"] = int(sub["FP"].sum())
        agg["FN"] = int(sub["FN"].sum())
        agg["TN"] = int(sub["TN"].sum())
        metrics = compute_metrics_from_counts(agg["TP"], agg["FP"], agg["FN"], agg["TN"])
        agg.update(metrics)
        domain_rows.append(agg)

df_domain = pd.DataFrame(domain_rows)
df_domain.to_csv(OUTPUT_DIR / "domain_level_aggregated_metrics.csv", index=False)
print("Domain-level aggregated metrics:")
df_domain

In [ ]:
# Domain-level bar chart
metric_labels = ["Accuracy", "Sensitivity", "Specificity", "PPV", "NPV", "Fab. Rate"]
metric_keys = ["accuracy", "sensitivity", "specificity", "ppv", "npv", "fabrication_rate"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
for ax_i, (domain_name, _) in enumerate(DOMAINS.items()):
    ax = axes[ax_i]
    dom = df_domain[df_domain["Domain"] == domain_name]
    h = dom[dom["Annotator"] == "Human"][metric_keys].values.flatten()
    a = dom[dom["Annotator"] == "AI"][metric_keys].values.flatten()
    x = np.arange(len(metric_keys))
    width = 0.35
    ax.bar(x - width/2, h, width, label="Human", color="#2c3e50")
    ax.bar(x + width/2, a, width, label="AI", color="#95a5a6")
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, rotation=35, ha="right")
    ax.set_ylim(0, 1.1)
    ax.set_title(domain_name, fontsize=14, fontweight="bold")
    ax.legend()
    ax.grid(axis="y", linestyle=":", alpha=0.6)

plt.suptitle("Domain-Level Metrics: Human vs AI", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "domain_aggregated_diagnostic_metrics_human_vs_ai.png", dpi=300, bbox_inches="tight")
plt.show()

---
## Part 4: Inference — McNemar P-Values

### 4.1 Element-Level P-Values Table

In [ ]:
# Extract p-values from element-level results
p_cols = [c for c in df_elements.columns if c.startswith("p_mcnemar_")]
df_pvals = df_elements[df_elements["Annotator"] == "Human"][["Element"] + p_cols].copy()
df_pvals.columns = [c.replace("p_mcnemar_", "") if c != "Element" else c for c in df_pvals.columns]

def sig_star(p):
    if pd.isna(p): return ""
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return ""

for col in df_pvals.columns:
    if col != "Element":
        df_pvals[f"{col}_star"] = df_pvals[col].apply(sig_star)

print("Element-Level One-Sided McNemar P-Values (H1: AI > Human)")
print("* p<0.05  ** p<0.01  *** p<0.001")
df_pvals

### 4.2 P-Values Stratified by Domain

In [ ]:
for domain_name, domain_elements in DOMAINS.items():
    print(f"\n{'=' * 60}")
    print(f"Domain: {domain_name}")
    print(f"{'=' * 60}")
    dom_pvals = df_pvals[df_pvals["Element"].isin(domain_elements)]
    for _, row in dom_pvals.iterrows():
        print(f"  {row['Element']:40s}", end="")
        for metric in ["accuracy", "sensitivity", "specificity", "ppv", "npv", "fabrication_rate"]:
            if metric in row:
                pv = row[metric]
                star = row.get(f"{metric}_star", "")
                print(f"  {metric[:6]}={pv:.4f}{star}", end="")
        print()

### 4.3 Fabrication Rate Focus: Element-Level

In [ ]:
fab_cols = ["Element", "Annotator", "FP", "TN", "fabrication_rate",
            "fabrication_rate_ci_low", "fabrication_rate_ci_high", "p_mcnemar_fabrication_rate"]
fab_available = [c for c in fab_cols if c in df_elements.columns]
df_fab = df_elements[fab_available].copy()
print("Fabrication Rate by Element & Annotator (with McNemar p-value H1: AI > Human)")
df_fab

In [ ]:
# Save all tables
df_pvals.to_csv(OUTPUT_DIR / "element_pvalues_one_sided.csv", index=False)
df_fab.to_csv(OUTPUT_DIR / "fabrication_rate_element_level.csv", index=False)
df_domain.to_csv(OUTPUT_DIR / "domain_level_aggregated_metrics.csv", index=False)

print("All tables and plots saved to:", OUTPUT_DIR)